# TROPESS Compression: Variable Decompression Example

This notebook demonstrates how to decompress individual variables from a compressed TROPESS full product netCDF file using the `tropess-compression` package.

The decompression process extracts compressed data into NumPy arrays without writing back to a file.

## Summary

This notebook demonstrated:
1. Opening a compressed TROPESS full product netCDF file
2. Finding variables with compressed dimensions
3. Decompressing individual variables using `Multiple_Sounding_Decompression`

The key steps for decompression are:
- Read the compressed byte array from the variable: `compressed_var[:]`
- Create a decompressor object: `Multiple_Sounding_Decompression(compressed_input)`
- Decompress to 3D array: `decompressor.decompress_3D()`

In [1]:
import re
import netCDF4
import numpy as np
from tropess_compression.akc_compression import Multiple_Sounding_Decompression

## Configuration

Define the input file and the regular expressions used to identify which variables were compressed.

In [ ]:
# Input compressed file
input_filename = 'TROPESS_CrIS-JPSS1_L2_Full_CH4_20240818_MUSES_R1p23_FS_F2p10_J0.nc'

# Regular expressions for identifying compressed variables
COMPRESS_DIMENSIONS_RE = r'.*_compressed_bytes'

## Find Compressed Variables

Search the netCDF file for variables that have a compressed dimension (ending with `_compressed_bytes`).

In [3]:
def find_compressed_variables(dataset):
    """Find all variables with compressed_bytes dimension."""
    compressed_vars = []
    
    def search_group(group, path=""):
        for var_name, var_obj in group.variables.items():
            if len(var_obj.dimensions) > 0 and re.match(COMPRESS_DIMENSIONS_RE, var_obj.dimensions[0]):
                full_path = f"{path}/{var_name}".lstrip('/')
                compressed_vars.append(full_path)
        
        for grp_name, grp_obj in group.groups.items():
            search_group(grp_obj, f"{path}/{grp_name}")
    
    search_group(dataset)
    return compressed_vars

In [4]:
# Open the compressed file
data_file = netCDF4.Dataset(input_filename, 'r')

# Find all compressed variables
compressed_variables = find_compressed_variables(data_file)

print(f"Found {len(compressed_variables)} compressed variable(s):")
for var in compressed_variables:
    print(f"  - {var}")

Found 5 compressed variable(s):
  - averaging_kernel
  - observation_error_covariance
  - characterization/measurement_error_covariance
  - characterization/prior_covariance
  - characterization/total_error_covariance


## Decompress a Single Variable

Demonstrate decompressing one variable into a NumPy array.

In [5]:
def decompress_variable_data(dataset, var_name):
    """Decompress a single variable and return the decompressed NumPy array."""
    
    # Access the compressed variable
    compressed_var = dataset[var_name]
    
    # Read compressed data
    compressed_input = compressed_var[:]
    
    # Get metadata about the variable
    uncompressed_dims = compressed_var.uncompressed_dimensions
    uncompressed_dtype = compressed_var.uncompressed_data_type
    uncompressed_fill_value = compressed_var.uncompressed_fill_value
    max_error = compressed_var.compression_max_error
    
    print(f"Variable: {var_name}")
    print(f"  Expected uncompressed dimensions: {uncompressed_dims}")
    print(f"  Expected uncompressed dtype: {uncompressed_dtype}")
    print(f"  Fill value: {uncompressed_fill_value}")
    print(f"  Compression max error: {max_error}")
    
    # Decompress the data
    decompressor = Multiple_Sounding_Decompression(compressed_input)
    decompressed_data = decompressor.decompress_3D()
    
    print(f"  Actual decompressed shape: {decompressed_data.shape}")
    print(f"  Actual decompressed dtype: {decompressed_data.dtype}")
    
    return decompressed_data

In [6]:
# Decompress the first variable as an example
if compressed_variables:
    example_var = compressed_variables[0]
    decompressed_array = decompress_variable_data(data_file, example_var)
else:
    print("No compressed variables found in the file.")

Variable: averaging_kernel
  Expected uncompressed dimensions: ['target', 'level_fm', 'level_fm']
  Expected uncompressed dtype: float32
  Fill value: -999.0
  Compression max error: 5e-05
  Actual decompressed shape: (41680, 67, 67)
  Actual decompressed dtype: float64


## Cleanup

Close the netCDF file when done.

In [7]:
data_file.close()